In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_13_try_0.pickle

In [3]:
%%RecordEvent
%%time
### cell 14 ###

# Preprocess 2018 and 2019 datasets without modifying the originals
orig_q = 'On which platforms have you begun or completed data science courses'
alt_q = 'On which online platforms have you begun or completed data science courses'
# Column renaming maps for 2018
col_map_2018 = {
    alt_q: orig_q,
    'Kaggle Learn': 'Kaggle Learn Courses',
    'Fast.AI': 'Fast.ai',
    'Online University Courses': 'University Courses (resulting in a university degree)'
}
# Value replacement maps for 2018
val_map_2018 = {
    'Kaggle Learn': 'Kaggle Learn Courses',
    'Fast.AI': 'Fast.ai',
    'Online University Courses': 'University Courses (resulting in a university degree)'
}
# Prepare 2018
df_2018_proc = responses_df_2018_cell10.copy()
for old, new in col_map_2018.items():
    df_2018_proc.columns = df_2018_proc.columns.str.replace(old, new, regex=False)
df_2018_proc = df_2018_proc.replace(val_map_2018)
# Prepare 2019
col_map_2019 = {'Kaggle Courses (i.e. Kaggle Learn)': 'Kaggle Learn Courses'}
val_map_2019 = col_map_2019
_df2019 = responses_df_2019_cell10.copy()
for old, new in col_map_2019.items():
    _df2019.columns = _df2019.columns.str.replace(old, new, regex=False)
df_2019_proc = _df2019.replace(val_map_2019)

# Optimized subset grabber
def grab_subset(df, question):
    cols = [c for c in df.columns if question in c]
    labels = [c.split('-')[-1].lstrip() for c in cols]
    return df.loc[:, cols].rename(columns=dict(zip(cols, labels))).dropna(how='all', subset=labels)

# Combine across years
def combine_data(question, include_2017=False):
    years = ['2018','2019','2020','2021','2022']
    dfs   = [df_2018_proc, df_2019_proc, responses_df_2020, responses_df_2021, responses_df_2022_cell10]
    if include_2017:
        years.insert(0, '2017'); dfs.insert(0, responses_df_2017)
    parts = [grab_subset(df, question).assign(year=yr) for df, yr in zip(dfs, years)]
    combined = pd.concat(parts, ignore_index=True)
    counts   = combined.groupby('year', as_index=False).count()
    return combined, counts

# Convert raw counts to percentages
def to_percentages(combined, counts):
    totals = combined['year'].value_counts()
    pct    = counts.set_index('year').divide(totals, axis=0) * 100
    return pct.reset_index()

# Run the pipeline
# replicate original final question string
question_of_interest_cell26 = orig_q + '?'  
learning_platform_df_combined, learning_platform_df_combined_counts = combine_data(question_of_interest_cell26)
learning_platform_df_combined_percentages = to_percentages(
    learning_platform_df_combined,
    learning_platform_df_combined_counts
)
# Clean up column names
learning_platform_df_combined_percentages.columns = (
    learning_platform_df_combined_percentages.columns
    .str.replace('(resulting in a university degree)', '', regex=False)
)
# Select and melt only the desired platforms
keep_cols = [
    'year', 'Coursera', 'University Courses ', 'Kaggle Learn Courses',
    'Udemy', 'Udacity', 'DataCamp', 'edX', 'Fast.ai', 'None', 'Other'
]
subset = learning_platform_df_combined_percentages.loc[:, keep_cols]
value_vars = [c for c in subset.columns if c not in ['year', 'None', 'Other']]
df_cell26 = (
    subset
    .melt(id_vars='year', value_vars=value_vars)
    .sort_values(['year', 'value'])
    .rename(columns={'variable': ''})
)
df_cell26.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40 entries, 35 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    40 non-null     object 
 1           40 non-null     object 
 2   value   40 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.2+ KB
CPU times: user 178 ms, sys: 4.35 ms, total: 182 ms
Wall time: 182 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_14_try_0.pickle